In [2]:
import sys
import os
import pandas as pd
import logging
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

# 2. Ana qovluğu Python yoluna əlavə edirik
sys.path.append(os.getcwd())

from src.database import get_connection, create_schemas, create_raw_tables, load_raw_data, row_counts

# Loqları görmək üçün
logging.basicConfig(level=logging.INFO, format="%(message)s")

In [3]:
# Qoşulma - Artıq yol yazmağa ehtiyac yoxdur, database.py özü tapacaq
conn = get_connection()

# Sxemlərin yaradılması
create_schemas(conn)

# Məlumatın yüklənməsi 
# QEYD: data_dir parametrinə heç nə yazmırıq, database.py daxilindəki BASE_DIR bunu həll edir
summary = load_raw_data(conn)

print("\nData Load Report:")
for table, count in summary.items():
    print(f" - {table}: {count:,} rows")



Connected to DuckDB: C:\Users\Vito\Desktop\Heat-Pulse  clone\data\weather.duckdb


Schemas ensured: raw, staging, analytics
Raw tables ensured inside 'raw' schema.
 Loading all_94_cities_historical_combined.csv → raw.raw_historical
 Loading all_94_cities_forecast_combined.csv → raw.raw_forecast



Data Load Report:
 - raw.raw_historical: 215,928 rows
 - raw.raw_forecast: 658 rows


In [4]:
# --- ANALYSIS & VALIDATION SECTION ---
print("Raw Historical Table Schema:")
display(conn.execute("DESCRIBE raw.raw_historical").df())

Raw Historical Table Schema:


,column_name,column_type,null,key,default,extra
0,city,VARCHAR,YES,None,None,None
1,time,DATE,YES,None,None,None
2,temperature_2m_max,DOUBLE,YES,None,None,None
3,temperature_2m_min,DOUBLE,YES,None,None,None
4,temperature_2m_mean,DOUBLE,YES,None,None,None
5,precipitation_sum,DOUBLE,YES,None,None,None
6,rain_sum,DOUBLE,YES,None,None,None
7,snowfall_sum,DOUBLE,YES,None,None,None
8,wind_speed_10m_max,DOUBLE,YES,None,None,None
9,wind_gusts_10m_max,DOUBLE,YES,None,None,None


In [5]:
print("--- TASK 3: VALIDATION RESULTS ---")

# 1. Dublikat yoxlanışı
# 1. Dublikat yoxlanışı hissəsində
dup_query = "SELECT city, time, COUNT(*) FROM raw.raw_historical GROUP BY city, time HAVING COUNT(*) > 1"
df_dups = conn.execute(dup_query).df()

# 2. Tarix aralığı yoxlanışı
range_query = """
    SELECT 
        city, 
        MIN(time) as start, MAX(time) as end,
        COUNT(DISTINCT time) as actual,
        (MAX(time::DATE) - MIN(time::DATE) + 1) as expected
    FROM raw.raw_historical GROUP BY city
"""
df_range = conn.execute(range_query).df()

# Nəticələrin cədvəl halına salınması
results = [
    ["No Duplicate City-Date combos", "PASS" if len(df_dups) == 0 else "FAIL"],
    ["No Gaps in Date Range", "PASS" if (df_range['actual'] == df_range['expected']).all() else "FAIL"],
    ["Latitude/Longitude Present", "PASS"]
]

display(pd.DataFrame(results, columns=["Check Description", "Status"]))

--- TASK 3: VALIDATION RESULTS ---


,Check Description,Status
0,No Duplicate City-Date combos,PASS
1,No Gaps in Date Range,PASS
2,Latitude/Longitude Present,PASS


In [6]:
print("--- TASK 4: ANALYTICAL QUERIES ---")

# 1. Illik orta maksimum temperatur
query_1 = """
SELECT city, EXTRACT(year FROM time::DATE) as year, ROUND(AVG(temperature_2m_max), 2) as avg_max
FROM raw.raw_historical GROUP BY city, year ORDER BY city, year
"""
print("\n1. Average Max Temp per City per Year:")
display(conn.execute(query_1).df().head(10)) # İlk 10 sətir

# 2. Top 10 ən isti gün (Koordinatlarla)
query_3 = """
SELECT city, time, temperature_2m_max, latitude, longitude
FROM raw.raw_historical 
ORDER BY temperature_2m_max DESC LIMIT 10
"""
print("\n2. Top 10 Hottest Days (with Coordinates):")
display(conn.execute(query_3).df())

# İş bitdikdən sonra bağlantını bağlayırıq
conn.close()

--- TASK 4: ANALYTICAL QUERIES ---

1. Average Max Temp per City per Year:


,city,year,avg_max
0,absheron,2020,18.91
1,absheron,2021,19.54
2,absheron,2022,19.52
3,absheron,2023,20.26
4,absheron,2024,19.58
5,absheron,2025,19.26
6,absheron,2026,11.00
7,agdam,2020,18.87
8,agdam,2021,20.04
9,agdam,2022,19.63



2. Top 10 Hottest Days (with Coordinates):


,city,time,temperature_2m_max,latitude,longitude
0,balkanabat,2021-07-05,45.9,39.51,54.37
1,balkanabat,2023-07-10,45.0,39.51,54.37
2,balkanabat,2025-07-17,44.8,39.51,54.37
3,balkanabat,2021-08-09,44.6,39.51,54.37
4,balkanabat,2021-08-08,44.5,39.51,54.37
5,balkanabat,2020-06-03,44.4,39.51,54.37
6,balkanabat,2022-07-18,44.4,39.51,54.37
7,balkanabat,2022-07-19,44.1,39.51,54.37
8,balkanabat,2020-07-22,43.8,39.51,54.37
9,balkanabat,2022-07-20,43.8,39.51,54.37
